# VM.AI Parser Training

**Enable GPU first:**
- Runtime → Change runtime type → GPU (T4) → Save

Uses **uv** for fast package installation.

In [ ]:
!pip install uv -q
print('uv installed')

In [ ]:
!uv add transformers datasets torch pyyaml huggingface_hub
print('Dependencies added with uv')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Auto-Download Latest Model

Set up Colab Secrets for automatic download:
- Click the 🔑 (secrets) icon in the left panel
- Add secret: `HF_TOKEN` = your Hugging Face token
- Run the cell below to auto-download the latest model

In [ ]:
import torch
print(f'GPU: {torch.cuda.is_available()}')
print(f'Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# Auto-download latest model from Hugging Face
# Set HF_TOKEN in Colab secrets or paste it below
from huggingface_hub import snapshot_download
import os

# Get token from Colab secrets (recommended) or paste directly
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except:
    HF_TOKEN = ''  # Will prompt for token if needed

os.makedirs('models/finetuned_parser', exist_ok=True)

print('Downloading latest model from Hugging Face...')
try:
    snapshot_download(
        repo_id='vaneaa/vmai-parser',
        local_dir='models/finetuned_parser',
        token=HF_TOKEN if HF_TOKEN else None
    )
    print('Model downloaded successfully')
except Exception as e:
    print(f'Download failed: {e}')
    print('Make sure repo exists and token is valid (if private)')

print('Ready')

In [ ]:
!python train.py --mode both

In [ ]:
# Save to Drive
!mkdir -p /content/drive/MyDrive/vmai/models
!cp -r ../models/finetuned_parser /content/drive/MyDrive/vmai/models/
print('Saved to Google Drive')

## Test Model (Chat Interface)

In [ ]:
from transformers import AutoTokenizer, T5ForConditionalGeneration
import torch
import json

model_path = '../models/finetuned_parser'
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print('Model loaded')

In [ ]:
def predict(input_text):
    inputs = tokenizer(input_text, return_tensors='pt', truncation=True, padding='max_length', max_length=256).to(device)
    outputs = model.generate(inputs['input_ids'], max_new_tokens=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_cases = [
    'add: gym at 6am',
    'add: meditate every morning',
    'add: finish report by Friday',
    'add: team meeting at 3pm',
]

for inp in test_cases:
    output = predict(inp)
    print(f'Input:  {inp}')
    print(f'Output: {output}')
    print()

In [ ]:
print('VM.AI Parser - Interactive Test')
print('Type your prompts (or "quit" to exit)')
print()

while True:
    user_input = input('Enter: ')
    if user_input.lower() == 'quit':
        break
    
    if not user_input.startswith('add:') and not user_input.startswith('modify:'):
        user_input = 'add: ' + user_input
    
    output = predict(user_input)
    print(f'Output: {output}')
    print()